# Trinity Finetuning — Plain PEFT + TRL (LoRA)

A more transparent, "from the fundamentals" LoRA finetuning walkthrough using
plain Hugging Face `transformers`, `peft`, and `trl` — no Unsloth speedups.
Useful for understanding what Unsloth is doing under the hood.

Runs on `meta-llama/Llama-3.2-3B-Instruct` (gated — requires an accepted HF
license and `huggingface-cli login`) on the same `cyber_defense_qa.jsonl`
dataset. Works in **Google Colab with a GPU runtime**; on this Mac (no CUDA)
it will run only on CPU, which is fine for a tiny smoke test but far too slow
for real training.


In [ ]:
%%capture
%pip install -U transformers accelerate peft trl bitsandbytes datasets


## 1. Configuration


In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # gated: accept license on HF + huggingface-cli login
DATASET_PATH = "cyber_defense_qa.jsonl"

OUTPUT_DIR = "trinity-peft-trl-lora"
MAX_STEPS = 60
LEARNING_RATE = 2e-4

LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

SYSTEM_PROMPT = (
    "You are Trinity, a cybersecurity defense assistant. You help with threat "
    "detection, incident response, secure coding, vulnerability analysis, and "
    "hardening systems. Be concise, precise, and security-conscious."
)


## 2. Load tokenizer, base model (4-bit), and dataset


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
)

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")


def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print(dataset)


## 3. Attach a LoRA adapter with `peft`


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 4. Train with `SFTTrainer`


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        bf16=torch.cuda.is_available(),
        seed=42,
    ),
)

trainer.train()


## 5. Save the adapter and run a sanity-check generation


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.eval()
test_question = "What should I check first when a server shows unexpected outbound traffic?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_question},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))
